In [57]:
import numpy
print(numpy.__version__)

2.5.1


## Projeto: "Convergência Estatística Aplicada à Precificação de Opções via Modelo Binomial"
Demonstrar empiricamente, através de simulação, como a LFGN, a LGGN e o TLC (via Teorema de De Moivre-Laplace) sustentam o método de precificação de opções por árvore binomial — e quantificar a incerteza da estimativa via Monte Carlo.

In [58]:
import numpy as np
import matplotlib.pyplot as mt
from scipy.stats import binom , norm
import yfinance as yf
import pandas as pd
import requests as rt
import math as m 

Obtenção dos dados que seram usados

In [59]:
dados = yf.download(
    "PETR4.SA",
    period="2y"
)



[*********************100%***********************]  1 of 1 completed


## Modulo  0: preparação e coleta dos dados

Calcular o Log-retorno diario

In [60]:
close  = dados['Close'].squeeze()
razao = close / close.shift(1) # razao entre o fechamento de hoje com o  dia anterior


log = np.log(razao) # calcula o log-retorno diario
log_retorno = log.dropna() 

dp = log_retorno.std() # desvio padrao
dp_anual = dp * np.sqrt(252) # volatilidade anual





print(dp_anual)



0.245550380703654


Puxando API do banco central

In [61]:
codigo = 11
n = 1
url = f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados/ultimos/{n}?formato=json'
resposta = rt.get(url)
selic = resposta.json()
r = float(selic[0]['valor']) # Taxa livre de risco



Extraindo os restante dos dados

In [62]:
s0 = close.iat[-1] # Preço Atual da ação
strike = round(s0) # Preço em exercicio, arredondado
T = 0.25 # tempo de vencimento do mercado
n_steps = 200 # Divide em quantos remos vai ter ate a data de vencimento


OUTPUTS

In [63]:

def outputs(vol):
    deltaT = T/n_steps
    u = m.exp(vol * m.sqrt(deltaT))
    d = 1/u
    p = (m.exp(r * deltaT) - d) / (u - d)

    return u,d, p

u, d, p = outputs(dp_anual)

print(u, d, p)


1.0087193106239307 0.9913560585862706 0.5016115241977666


Função  simuladora de trajetoria

In [69]:
# Achando o valor de k 

def simulacao(s, P, sub, des, si, K_strike):
    k = np.random.binomial(s, P)
    Sn = si * (sub ** k) * (des ** (s - k))
    payoff = max(Sn - K_strike, 0)

    return Sn, k, payoff

Sn, k, payoff = simulacao(n_steps, p, u, d, s0, strike)

print(Sn, k, payoff)


49.258233295037364 108 6.258233295037364


## Calculando o preço teorico


In [72]:
esperanca = 0

for k in range(201):
    probk = binom.pmf(k, n_steps, p)  # probabilidade de k altas

    Snk = s0 * (u ** k) * (d ** (n_steps - k))

    payoffk = max(Snk - strike, 0)

    contribuicao = probk * payoffk

    esperanca += contribuicao


def teorico(j, t, e):
    C = np.exp(-j * t) * e
    return C


preco_teorico = teorico(r, T, esperanca)

print(preco_teorico)




2.31084496923271


## Modulo 2: Lei Fraca dos Grandes Numeros

# Simulação 

In [73]:
payoff_media = []

for m in range(1001):
    _, _, pyf= simulacao(n_steps, p, u, d, s0, strike)
    payoff_media.append(float(pyf))

print(payoff_media)

[1.3848557062899403, 2.9532415477382727, 2.9532415477382727, 4.5770479624545, 0.0, 0.0, 4.5770479624545, 8.8920515410407, 0.0, 1.3848557062899403, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.800922538352438, 0.0, 2.9532415477382727, 0.0, 0.0, 0.0, 0.0, 0.0, 4.5770479624545, 2.9532415477382727, 0.0, 0.0, 0.0, 0.0, 0.0, 8.8920515410407, 0.0, 4.5770479624545, 1.3848557062899403, 2.9532415477382727, 2.9532415477382727, 0.0, 0.0, 0.0, 4.5770479624545, 0.0, 0.0, 7.998825090264191, 3.758096380701815, 0.0, 0.0, 2.9532415477382727, 1.3848557062899403, 5.4103431925840155, 7.998825090264191, 3.758096380701815, 0.0, 0.0, 3.758096380701815, 0.0, 0.0, 0.0, 0.6208518568872705, 0.0, 0.0, 17.669004557705925, 0.0, 9.800922538352438, 0.0, 7.120973893860423, 0.6208518568872705, 5.4103431925840155, 0.0, 0.6208518568872705, 1.3848557062899403, 0.0, 0.0, 0.0, 6.258233295037364, 0.0, 0.6208518568872705, 3.758096380701815, 1.3848557062899403, 0.0, 0.0, 0.0, 0.6208518568872705, 0.0, 0.0, 3.758096380701815, 0.0, 14.58970577